# Servicio de scoring crediticio

In [8]:
from pathlib import Path
from flask import Flask, request, jsonify
import pandas as pd
import numpy as np
import joblib
import shap
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")
load_dotenv() #fichero.env

True

## 1. Cargo el modelo y el clustering

In [ ]:
RUTA_MODELO = Path("models/modelo_scoring_final.joblib")
RUTA_CLUSTER = Path("models/clustering_perfiles.joblib")
modelo = joblib.load(RUTA_MODELO)
clustering = joblib.load(RUTA_CLUSTER)
COLUMNAS_MODELO = list(modelo.feature_names_in_)
TASA_GLOBAL = 0.224

#Segmentos clustering
NOMBRES_SEGMENTOS = {
    0: "Deuda e importe bajos",
    1: "Ingresos bajos y DTI alto",
    2: "Ingresos e importe altos",
    3: "FICO alto",
}

print("Modelo:", modelo.named_steps["clf"].__class__.__name__)
print("Variables:", COLUMNAS_MODELO)
print("Segmentos:", NOMBRES_SEGMENTOS)

Modelo: LGBMClassifier
Variables: ['revenue', 'dti_n', 'loan_amnt', 'fico_n', 'emp_length', 'purpose', 'home_ownership_n', 'addr_state']
Segmentos: {0: 'Deuda e importe bajos', 1: 'Ingresos bajos y DTI alto', 2: 'Ingresos e importe altos', 3: 'FICO alto'}


## 2. Preparo el explicador SHAP

In [ ]:

preprocesador = modelo.named_steps["pre"]
clasificador = modelo.named_steps["clf"]
explainer_shap = shap.TreeExplainer(clasificador)

CATEGORICAS = ["emp_length", "purpose", "home_ownership_n", "addr_state"]
def nombre_legible(nombre):
    base = nombre.split("__")[-1]
    for columna in CATEGORICAS:
        prefijo = columna + "_"
        if base.startswith(prefijo):
            valor = base[len(prefijo):]
            return f"{columna} = {valor}"

    return base
nombres_features = [nombre_legible(n) for n in preprocesador.get_feature_names_out()]

## 3. Funciones del servicio

In [11]:
NUMERICAS = ["revenue", "dti_n", "loan_amnt", "fico_n"]

RANGOS_VALIDOS = [
    ("revenue", 0, None, "revenue debe ser un numero no negativo."),
    ("loan_amnt", 1000, 40000, "loan_amnt debe estar entre 1.000 y 40.000 dolares."),
    ("fico_n", 662, 850, "fico_n debe estar entre 662 y 850 puntos."),
    ("dti_n", 0, None, "dti_n no puede ser negativo."),
]


def preparar_entrada(data):
    """Función que valida el JSON recibido y lo deja con el esquema que espera el pipeline."""
    if not isinstance(data, dict):
        raise ValueError("El cuerpo de la peticion debe ser un objeto JSON.")
    faltan = [c for c in COLUMNAS_MODELO if c not in data]
    if faltan:
        raise ValueError(f"Faltan variables obligatorias: {faltan}")
    fila = pd.DataFrame([{c: data[c] for c in COLUMNAS_MODELO}])
    for columna in NUMERICAS:
        fila[columna] = pd.to_numeric(fila[columna], errors="coerce")

    fila["dti_n"] = fila["dti_n"].replace(999, np.nan)
    if fila["revenue"].isna().any():
        raise ValueError("revenue debe ser un numero no negativo.")

    for columna, minimo, maximo, mensaje in RANGOS_VALIDOS:
        valores = fila[columna].dropna()
        si_hay_maximo = maximo is None or (valores <= maximo).all()
        if (valores < minimo).any() or not si_hay_maximo:
            raise ValueError(mensaje)

    return fila[COLUMNAS_MODELO]


def predecir_impago(data):
    """Devuelvo la probabilidad estimada, sin aplicar ningun umbral."""
    fila = preparar_entrada(data)
    return {
        "probabilidad_impago": round(float(modelo.predict_proba(fila)[0, 1]), 4),
        "tasa_impago_muestra": TASA_GLOBAL,
    }


def explicar_prediccion(data, top_n=6):
    """Anado a la prediccion los factores SHAP que mas pesan en este caso."""
    fila = preparar_entrada(data)
    resultado = predecir_impago(data)
    explicacion = explainer_shap(preprocesador.transform(fila))
    if explicacion.values.ndim == 3:
        explicacion = explicacion[..., 1]

    contribuciones = pd.Series(explicacion.values[0], index=nombres_features)
    contribuciones = contribuciones.sort_values(key=abs, ascending=False).head(top_n)

    resultado["factores"] = [
        {"variable": nombre,
         "efecto_shap": round(float(valor), 4),
         "direccion": "aumenta el riesgo" if valor > 0 else "reduce el riesgo"}
        for nombre, valor in contribuciones.items()
    ]
    return resultado


def asignar_cluster(data):
    """Asigno el caso a uno de los cuatro perfiles del capitulo de clustering."""
    fila = preparar_entrada(data)
    variables = clustering["variables"]
    entrada = fila[variables].copy()

    if entrada.isna().any().any():
        raise ValueError("La segmentacion necesita valores numericos completos.")

    for variable, transformacion in clustering.get("transformaciones", {}).items():
        if transformacion == "log1p":
            entrada[variable] = np.log1p(entrada[variable])

    id_cluster = int(clustering["kmeans"].predict(clustering["scaler"].transform(entrada))[0])
    perfil = clustering["perfil"].loc[id_cluster]

    # El artefacto guarda la tasa en porcentaje (26.5); la paso a proporcion.
    return {
        "cluster": id_cluster,
        "nombre_segmento": NOMBRES_SEGMENTOS[id_cluster],
        "tasa_impago_segmento": round(float(perfil["tasa_impago"]) / 100, 4),
        "medianas_segmento": {v: float(perfil[v]) for v in variables},
    }

## 5. Agente explicativo



In [ ]:
import json
import os

### Herramientas


In [13]:
HERRAMIENTAS = [
    {
        "type": "function",
        "function": {
            "name": "consultar_shap",
            "description": ("Devuelve la probabilidad estimada de impago del caso y los "
                            "factores que mas influyen en esa estimacion, ordenados por peso."),
            "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "consultar_segmento",
            "description": ("Devuelve el perfil de cartera al que pertenece el cliente y la "
                            "tasa historica de impago de ese segmento."),
            "parameters": {"type": "object", "properties": {}},
        },
    },
]

PROMPT_SISTEMA = """Eres un analista de riesgo crediticio que apoya a un equipo de riesgos bancario. Un modelo de aprendizaje automatico estima la probabilidad de que un solicitante no devuelva su prestamo. Quien te lee es un analista humano que necesita entender en segundos por que el modelo ha dado ese resultado.

Tienes dos herramientas: consultar_shap (que factores explican la estimacion de este caso) y consultar_segmento (a que perfil de cliente se parece). Usa las que necesites antes de responder.

Escribe un maximo de 6 frases, en espanol claro y sin tecnicismos. Reglas:
- Nada de jerga tecnica: habla de "los datos que mas influyen", no de SHAP ni de modelos.
- No digas si se aprueba o se deniega: la decision es del analista, tu solo explicas.
- Usa solo las cifras que te devuelvan las herramientas, no inventes ninguna.
"""

### Bucle ReAct

In [ ]:
from openai import OpenAI

MODELO_AGENTE = "gpt-4o-mini"


def ejecutar_herramienta(nombre, data):
    if nombre == "consultar_shap":
        return explicar_prediccion(data)
    if nombre == "consultar_segmento":
        return asignar_cluster(data)
    raise ValueError(f"Herramienta desconocida: {nombre}")


def generar_explicacion_agente(data, max_pasos=4):
    """Bucle ReAct con el modelo de lenguaje."""
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        return {"explicacion": "No se pudo generar la explicacion: falta la clave de API.", "generado_por": "error"}

    try:
        cliente = OpenAI(api_key=api_key)
        mensajes = [
            {"role": "system", "content": PROMPT_SISTEMA},
            {"role": "user", "content": f"Datos de la solicitud: {json.dumps(data, ensure_ascii=False)}"},
        ]

        for _ in range(max_pasos):
            respuesta = cliente.chat.completions.create(
                model=MODELO_AGENTE, messages=mensajes, tools=HERRAMIENTAS,
            )
            mensaje = respuesta.choices[0].message

            # Si ya no pide mas herramientas, ha terminado de razonar.
            if not mensaje.tool_calls:
                return {"explicacion": mensaje.content.strip(), "generado_por": MODELO_AGENTE}

            mensajes.append(mensaje)
            for llamada in mensaje.tool_calls:
                resultado = ejecutar_herramienta(llamada.function.name, data)
                mensajes.append({
                    "role": "tool",
                    "tool_call_id": llamada.id,
                    "content": json.dumps(resultado, ensure_ascii=False),
                })

        return {"explicacion": "No se pudo generar la explicacion: se agotaron los pasos del agente.", "generado_por": "error"}
    except Exception:
        return {"explicacion": "No se pudo generar la explicacion: fallo en la llamada al modelo de lenguaje.", "generado_por": "error"}

## 6. Rutas de la API


In [15]:
app = Flask(__name__)


@app.errorhandler(ValueError)
def error_validacion(error):
    return jsonify({"error": str(error)}), 400


@app.get("/salud")
def ruta_salud():
    return jsonify({
        "estado": "operativo",
        "modelo": clasificador.__class__.__name__,
        "n_variables": len(COLUMNAS_MODELO),
    })


@app.post("/predecir")
def ruta_predecir():
    return jsonify(predecir_impago(request.get_json(silent=True)))


@app.post("/explicar")
def ruta_explicar():
    return jsonify(explicar_prediccion(request.get_json(silent=True)))


@app.post("/cluster")
def ruta_cluster():
    return jsonify(asignar_cluster(request.get_json(silent=True)))


@app.post("/agente_explicacion")
def ruta_agente_explicacion():
    return jsonify(generar_explicacion_agente(request.get_json(silent=True)))


## 7. Levanto el servicio



In [16]:
if __name__ == "__main__":
    app.run(host="127.0.0.1", port=8080, debug=False)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:8080
Press CTRL+C to quit
